In [2]:
!pip install tensorflow==2.2
!pip install netCDF4
!pip install pysolar

     |████████████████████████████████| 516.2MB 24kB/s 
     |████████████████████████████████| 460kB 50.6MB/s 
     |████████████████████████████████| 3.0MB 53.5MB/s 
  Found existing installation: tensorflow-estimator 2.4.0
    Uninstalling tensorflow-estimator-2.4.0:
      Successfully uninstalled tensorflow-estimator-2.4.0
  Found existing installation: tensorboard 2.4.1
    Uninstalling tensorboard-2.4.1:
      Successfully uninstalled tensorboard-2.4.1
  Found existing installation: tensorflow 2.4.1
    Uninstalling tensorflow-2.4.1:
      Successfully uninstalled tensorflow-2.4.1


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from google.colab import files

import tensorflow as tf

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.layers import Input, Dense, Dropout, Reshape, Flatten, \
    LSTM, GRU, Concatenate, BatchNormalization, TimeDistributed, RepeatVector
from tensorflow.keras.models import Model, load_model

from tensorflow.keras import metrics, losses
from tensorflow.keras import backend as K

tf.test.gpu_device_name()
print(tf.__version__)

2.2.0


In [3]:
from datetime import datetime, timedelta
from netCDF4 import Dataset

from google.colab import drive, files
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [7]:
# Model Data
ncfid = Dataset('/content/gdrive/My Drive/Colab Notebooks/ERA5ViennaAdditionalFeatures.nc', mode='r') 

# Obs Data
obs_data_vis = np.load('/content/gdrive/My Drive/Colab Notebooks/VisobsFull.npy', allow_pickle=True)
obs_data_T = np.load('/content/gdrive/My Drive/Colab Notebooks/TobsFull.npy', allow_pickle=True)
obs_data_Td = np.load('/content/gdrive/My Drive/Colab Notebooks/TdobsFull.npy', allow_pickle=True)
sza = np.load('/content/gdrive/My Drive/Colab Notebooks/sza.npy', allow_pickle=True)

In [ ]:
# Get model data
t2m = ncfid['tas'][0,:,:,:]-273.15
d2m = ncfid['dpt'][0,:,:,:]-273.15
p2m = ncfid['ps'][0,:,:,:]
u2m = ncfid['uas'][0,:,:,:]
v2m = ncfid['vas'][0,:,:,:]
lat = ncfid['lat'][:]
lon = ncfid['lon'][:]

In [ ]:
# Get model data
model_data_T = np.column_stack((t2m[:,0,0], t2m[:,0,-1], t2m[:,-1,-1], t2m[:,-1,0]))
model_data_Td = np.column_stack((d2m[:,0,0], d2m[:,0,-1], d2m[:,-1,-1], d2m[:,-1,0]))
model_data_p = np.column_stack((p2m[:,0,0], p2m[:,0,-1], p2m[:,-1,-1], p2m[:,-1,0]))
model_data_u = np.column_stack((u2m[:,0,0], u2m[:,0,-1], u2m[:,-1,-1], u2m[:,-1,0]))
model_data_v = np.column_stack((v2m[:,0,0], v2m[:,0,-1], v2m[:,-1,-1], v2m[:,-1,0]))

In [ ]:
# Duplicate SZA information
sza = np.repeat(np.expand_dims(sza, -1), 4, axis=1)

In [ ]:
# Remove extraneous obs data
obs_data_T = obs_data_T[:-7]
obs_data_Td = obs_data_Td[:-7]
obs_data_vis = obs_data_vis[:-7]

In [ ]:
max_vis = obs_data_vis != obs_data_vis.max()
model_data_T = model_data_T[max_vis,]
model_data_Td = model_data_Td[max_vis,]
model_data_p = model_data_p[max_vis,]
model_data_u = model_data_u[max_vis,]
model_data_v = model_data_v[max_vis,]

obs_data_T = obs_data_T[max_vis,]
obs_data_Td = obs_data_Td[max_vis,]
obs_data_vis = obs_data_vis[max_vis,]
sza = sza[max_vis,]

IndexError: ignored

In [ ]:
# Bin the data
bins = [0, 150, 350, 600, 800, 1500, 3000, 5000, 10000]
obs_data_vis = pd.cut(obs_data_vis, bins, labels=[0,1,2,3,4,5,6,7])

In [ ]:
datetime

In [ ]:
# # # Get solar zenith angle

# # # Start time of dataset
base = datetime.datetime(2010,1,1,0,0, tzinfo=timezone.utc) + datetime.timedelta(hours=1)

arr = [base + datetime.timedelta(hours=i) for i in range(len(obs_data_vis))]
sza = np.array([90.0 - get_altitude(48.2082, 16.3738, arr[i]) for i in range(len(obs_data_vis))])
print(sza.shape)
np.save('sza.npy',sza)


In [ ]:
# Use the past_history nr of data to predict future_target nr of data
past_history = 3
future_target = 3

In [ ]:
# Split into training data
TRAIN_SPLIT = np.int(0.8*len(obs_data_vis))
print(TRAIN_SPLIT)

In [ ]:
def multivariate_data_model(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    indices = range(i-history_size, i+target_size, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)


def multivariate_data_obs(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    indices = range(i-history_size, i, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)

In [ ]:
STEP = 1

# This is the visibility (target) dataset
x_train_vis, y_train_vis = multivariate_data_obs(obs_data_vis, obs_data_vis, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_vis, y_val_vis = multivariate_data_obs(obs_data_vis, obs_data_vis,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

# These are the auxiliary (T, Td) datasets
x_train_T_obs, _ = multivariate_data_obs(obs_data_T, obs_data_T, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_T_obs, _ = multivariate_data_obs(obs_data_T, obs_data_T,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_Td_obs, _ = multivariate_data_obs(obs_data_Td, obs_data_Td, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_Td_obs, _ = multivariate_data_obs(obs_data_Td, obs_data_Td,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_T_model, _ = multivariate_data_model(model_data_T, model_data_T, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_T_model, _ = multivariate_data_model(model_data_T, model_data_T,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_Td_model, _ = multivariate_data_model(model_data_Td, model_data_Td, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_Td_model, _ = multivariate_data_model(model_data_Td, model_data_Td,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_p_model, _ = multivariate_data_model(model_data_p, model_data_p, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_p_model, _ = multivariate_data_model(model_data_p, model_data_p,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_u_model, _ = multivariate_data_model(model_data_u, model_data_u, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_u_model, _ = multivariate_data_model(model_data_u, model_data_u,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_v_model, _ = multivariate_data_model(model_data_v, model_data_v, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_v_model, _ = multivariate_data_model(model_data_v, model_data_v,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

x_train_sza, _ = multivariate_data_model(sza, model_data_Td, 0,
                                                 TRAIN_SPLIT, past_history,
                                                 future_target, STEP)
x_val_sza, _ = multivariate_data_model(sza, model_data_Td,
                                             TRAIN_SPLIT, None, past_history,
                                             future_target, STEP)

In [ ]:
# Keep only 2.5% of cases where there has been a constant (max) visibility
idx = np.sum(y_train_vis, axis=1) != np.max(np.sum(y_train_vis, axis=1))
idx_2 = np.sum(y_train_vis, axis=1) == np.max(np.sum(y_train_vis, axis=1))

x_train_vis_1 = x_train_vis[idx,];  x_train_vis_2 = x_train_vis[idx_2,]
n_c = np.int(0.01*len(x_train_vis_2))

y_train_vis_1 = y_train_vis[idx,]; y_train_vis_2 = y_train_vis[idx_2,]

x_train_Td_obs_1 = x_train_Td_obs[idx,]; x_train_Td_obs_2 = x_train_Td_obs[idx_2,]
x_train_T_obs_1 = x_train_T_obs[idx,]; x_train_T_obs_2 = x_train_T_obs[idx_2,]

x_train_Td_model_1 = x_train_Td_model[idx,]; x_train_Td_model_2 = x_train_Td_model[idx_2,]
x_train_T_model_1 = x_train_T_model[idx,]; x_train_T_model_2 = x_train_T_model[idx_2,]

x_train_sza_1 = x_train_sza[idx,]; x_train_sza_2 = x_train_sza[idx_2,]

x_train_vis = np.concatenate((x_train_vis_1, x_train_vis_2[:n_c]))
y_train_vis = np.concatenate((y_train_vis_1, y_train_vis_2[:n_c]))

x_train_Td_obs = np.concatenate((x_train_Td_obs_1, x_train_Td_obs_2[:n_c]))
x_train_T_obs = np.concatenate((x_train_T_obs_1, x_train_T_obs_2[:n_c]))
x_train_Td_model = np.concatenate((x_train_Td_model_1, x_train_Td_model_2[:n_c]))
x_train_T_model = np.concatenate((x_train_T_model_1, x_train_T_model_2[:n_c]))
x_train_sza = np.concatenate((x_train_sza_1, x_train_sza_2[:n_c]))

In [ ]:
# # Same for validation
idx = np.sum(y_val_vis, axis=1) != np.max(np.sum(y_val_vis, axis=1))
idx_2 = np.sum(y_val_vis, axis=1) == np.max(np.sum(y_val_vis, axis=1))

x_val_vis_1 = x_val_vis[idx,];  x_val_vis_2 = x_val_vis[idx_2,]
n_c = np.int(0.01*len(x_val_vis_2))

y_val_vis_1 = y_val_vis[idx,]; y_val_vis_2 = y_val_vis[idx_2,]

x_val_Td_obs_1 = x_val_Td_obs[idx,]; x_val_Td_obs_2 = x_val_Td_obs[idx_2,]
x_val_T_obs_1 = x_val_T_obs[idx,]; x_val_T_obs_2 = x_val_T_obs[idx_2,]

x_val_Td_model_1 = x_val_Td_model[idx,]; x_val_Td_model_2 = x_val_Td_model[idx_2,]
x_val_T_model_1 = x_val_T_model[idx,]; x_val_T_model_2 = x_val_T_model[idx_2,]

x_val_sza_1 = x_val_sza[idx,]; x_val_sza_2 = x_val_sza[idx_2,]

x_val_vis = np.concatenate((x_val_vis_1, x_val_vis_2[:n_c]))
y_val_vis = np.concatenate((y_val_vis_1, y_val_vis_2[:n_c]))

x_val_Td_obs = np.concatenate((x_val_Td_obs_1, x_val_Td_obs_2[:n_c]))
x_val_T_obs = np.concatenate((x_val_T_obs_1, x_val_T_obs_2[:n_c]))
x_val_Td_model = np.concatenate((x_val_Td_model_1, x_val_Td_model_2[:n_c]))
x_val_T_model = np.concatenate((x_val_T_model_1, x_val_T_model_2[:n_c]))
x_val_sza = np.concatenate((x_val_sza_1, x_val_sza_2[:n_c]))

In [ ]:
plt.hist(x_train_vis[:,0],bins=8)

In [ ]:
# Using T-Td as feature
x_train_TTd_obs = x_train_T_obs-x_train_Td_obs
x_val_TTd_obs = x_val_T_obs-x_val_Td_obs

x_train_TTd_model = x_train_T_model-x_train_Td_model
x_val_TTd_model = x_val_T_model-x_val_Td_model

In [ ]:
# Normalize the feature
TTd_max = np.max(x_train_TTd_obs)
TTd_min = np.min(x_train_TTd_obs)
T_max = np.max(x_train_T_model) 
T_min = np.min(x_train_T_model) 
p_max = np.max(x_train_p_model) 
p_min = np.min(x_train_p_model) 
u_max = np.max(x_train_u_model) 
u_min = np.min(x_train_u_model) 
v_max = np.max(x_train_v_model) 
v_min = np.min(x_train_v_model) 

sza_max = np.max(x_train_sza) 
sza_min = np.min(x_train_sza) 

x_train_TTd_obs = (x_train_TTd_obs - TTd_min)/(TTd_max - TTd_min)
x_val_TTd_obs = (x_val_TTd_obs - TTd_min)/(TTd_max - TTd_min)

x_train_TTd_model = (x_train_TTd_model - TTd_min)/(TTd_max - TTd_min)
x_val_TTd_model = (x_val_TTd_model - TTd_min)/(TTd_max - TTd_min)

x_train_T_model = (x_train_T_model - T_min)/(T_max - T_min)
x_val_T_model = (x_val_T_model - T_min)/(T_max - T_min)

x_train_p_model = (x_train_p_model - p_min)/(p_max - p_min)
x_val_p_model = (x_val_p_model - p_min)/(p_max - p_min)

x_train_u_model = (x_train_u_model - u_min)/(u_max - u_min)
x_val_u_model = (x_val_u_model - u_min)/(u_max - u_min)

x_train_v_model = (x_train_v_model - v_min)/(v_max - v_min)
x_val_v_model = (x_val_v_model - v_min)/(v_max - v_min)

x_train_sza = (x_train_sza - sza_min)/(sza_max - sza_min)
x_val_sza = (x_val_sza - sza_min)/(sza_max - sza_min)

In [ ]:
# # Concatenate the data
# x_train_model = np.concatenate((x_train_T_model, x_train_TTd_model, x_train_p_model,
#                                 x_train_u_model, x_train_v_model, x_train_sza), axis=2)

# x_val_model = np.concatenate((x_val_T_model, x_val_TTd_model, x_val_p_model,
#                               x_val_u_model, x_val_v_model, x_val_sza), axis=2)

# x_train_obs = np.stack((x_train_vis, x_train_TTd_obs, x_train_T_obs), axis=2)
# x_val_obs = np.stack((x_val_vis, x_val_TTd_obs, x_val_T_obs), axis=2)

In [ ]:
# Concatenate the data
x_train_model = np.concatenate((x_train_T_model, x_train_TTd_model, x_train_p_model,
                                x_train_sza), axis=2)

x_val_model = np.concatenate((x_val_T_model, x_val_TTd_model, x_val_p_model,
                              x_val_sza), axis=2)

x_train_obs = np.stack((x_train_vis, x_train_TTd_obs, x_train_T_obs), axis=2)
x_val_obs = np.stack((x_val_vis, x_val_TTd_obs, x_val_T_obs), axis=2)

In [ ]:
# Total number of model features
nr_features_model = x_train_model.shape[2]
nr_features_obs = x_train_obs.shape[2]

In [ ]:
# Data shape
print ('Single window of past history : {}'.format(x_train_model[0].shape))
print ('Target visibility to predict : {}'.format(y_train_vis[0].shape))

In [ ]:
def multi_layer_cross_entropy(y_true, y_pred):

  t = np.ones(future_target)

  loss = t[0]*tf.keras.losses.sparse_categorical_crossentropy(y_true[:,0], y_pred[:,0,:])

  for i in range(1, future_target):
    y_true_step_i = y_true[:,i]
    y_pred_step_i = y_pred[:,i,:]
    loss += t[i]*tf.keras.losses.sparse_categorical_crossentropy(y_true_step_i, 
                                                              y_pred_step_i)
  return loss


def rps_loss(y_true, y_pred):

  rps = 0
  for i in range(future_target):

    y_pred_i = y_pred[:,i,:]
    y_true_i = y_true[:,i,:]

    for s in range(1,9):
      for j in range(1,s):
        rps += K.square(K.sum(y_true_i[:,:j] - y_pred_i[:,:j], axis=1))
        
    rps /= 7.0

  return K.mean(rps)

In [ ]:
input_shape_model = (past_history+future_target, nr_features_model,)
input_shape_obs = (past_history, nr_features_obs,)

# Obs branch
in_obs = Input(shape=input_shape_obs)
b_obs = LSTM(128, return_sequences=False, recurrent_dropout=0.0, 
                   dropout=0.)(in_obs)

# Model branch
in_model = Input(shape=input_shape_model)
b_model = LSTM(128, return_sequences=False, recurrent_dropout=0.0, 
                   dropout=0.)(in_model)               

# Now concatenate with all branches
b = Concatenate()([b_obs, b_model])
b = Dense(128*future_target, activation='selu')(b)
b = Dropout(0.5)(b)
b = Reshape((future_target, 128))(b)
b = LSTM(128, return_sequences=True, recurrent_dropout=0.0)(b)
out = TimeDistributed(Dense(8, activation='softmax'))(b)

model = Model([in_obs, in_model], out)
model.compile(optimizer='adam', loss=multi_layer_cross_entropy,
              metrics=['acc'])
model.summary()

In [ ]:
multi_step_history = model.fit([x_train_obs, x_train_model], 
                               y_train_vis, batch_size=256,
                               validation_split=0.05, epochs=40,
                               callbacks=tf.keras.callbacks.EarlyStopping(patience=5))

In [ ]:
def plot_train_history(history, title):
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  epochs = range(len(loss))

  plt.figure()

  plt.plot(epochs, loss, 'b', label='Training loss')
  plt.plot(epochs, val_loss, 'r', label='Validation loss')
  plt.title(title)
  plt.legend()

  plt.show()

In [ ]:
plot_train_history(multi_step_history, 'Multi-Step Training and validation loss')

In [ ]:
# Plot some results
ix = np.random.randint(0, len(x_val_vis))

vis = np.expand_dims(x_val_vis[ix,], axis=0)
true_vis = np.expand_dims(y_val_vis[ix,], axis=0)
data_obs = np.expand_dims(x_val_obs[ix,], axis=0)
data_model = np.expand_dims(x_val_model[ix,], axis=0)
# sza = np.expand_dims(x_val_sza[ix,], axis=0)

runs = 20
vis_prob = np.stack([model([data_obs, data_model], training=True) \
                     for sample in range(runs)])
vis_prob = np.argmax(vis_prob, axis=3)

vis_mean = np.round(vis_prob.mean(axis=0))
vis_std = vis_prob.std(axis=0)

# This is the input observations
plt.plot(np.arange(-past_history+1, 1), vis[0,:past_history],'k--')

# Persitence prediction
plt.plot(np.arange(1, future_target+1), x_val_vis[ix, past_history-1]*np.ones(future_target),'g.')

# ML prediction
plt.plot(np.arange(1, future_target+1), vis_mean[0,],'b')
plt.fill_between(np.arange(1, future_target+1), vis_mean[0,]-vis_std[0,], vis_mean[0,]+vis_std[0,])

# True observed future
plt.plot(np.arange(1, future_target+1), true_vis[0,],'r')

plt.legend(['Observed past','Persistence','ML model','Observed']) 
plt.grid()
plt.xlabel('time')
plt.ylabel('class')

# for i in range(runs):
#  plt.plot(np.arange(0, future_target), np.round(vis_prob[i,0,]),'r-.')

In [ ]:
res = model([data_obs, data_model], training=True)[0,].numpy()
print(res.sum(axis=1))

In [ ]:
# Proper verification
from sklearn.metrics import *

In [ ]:
# Now verify the model
n_val = len(x_val_vis)
y_pers = np.zeros((n_val, future_target))
y_ml = np.zeros((n_val, future_target))

err_pers = np.zeros(n_val)
err_ml = np.zeros(n_val)

for i in range(n_val):

  # 1) Persistence error
  y_pers[i,] = x_val_vis[i, past_history-1]*np.ones(future_target)
  err_pers[i] = np.mean(np.square(y_pers[i,] - y_val_vis[i,]))

  # 3) ML model error
  vis = np.expand_dims(x_val_vis[i,], axis=0)
  true_vis = np.expand_dims(y_val_vis[i,], axis=0)
  data_obs = np.expand_dims(x_val_obs[i,], axis=0)
  data_model = np.expand_dims(x_val_model[i,], axis=0)
  # sza = np.expand_dims(x_val_sza[i,], axis=0)
  vis_ml = model([data_obs, data_model])
  vis_ml = np.argmax(vis_ml, axis=2)
  
  err_ml[i] = np.mean(np.square(vis_ml[0,] - y_val_vis[i,]))
  y_ml[i,:] = vis_ml[0,]

print('Persistence error: {}'.format(np.sum(err_pers)))
print('ML model error: {}'.format(np.sum(err_ml)))

In [ ]:
nr_classes = 8

precision_pers = np.zeros((nr_classes,future_target))
recall_pers = np.zeros((nr_classes,future_target))
f1_pers = np.zeros((nr_classes,future_target))
balanced_acc_pers = np.zeros(future_target)

precision_ml = np.zeros((nr_classes,future_target))
recall_ml = np.zeros((nr_classes,future_target))
f1_ml = np.zeros((nr_classes,future_target))
balanced_acc_ml = np.zeros(future_target)

for i in range(future_target):
  precision_pers[:,i] = precision_score(y_val_vis[:,i], y_pers[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)
  precision_ml[:,i] = precision_score(y_val_vis[:,i], y_ml[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)

  recall_pers[:,i] = recall_score(y_val_vis[:,i], y_pers[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)
  recall_ml[:,i] = recall_score(y_val_vis[:,i], y_ml[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)

  f1_pers[:,i] = f1_score(y_val_vis[:,i], y_pers[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)
  f1_ml[:,i] = f1_score(y_val_vis[:,i], y_ml[:,i], 
                                        labels=[0,1,2,3,4,5,6,7], average=None)
  
  balanced_acc_pers[i] = balanced_accuracy_score(y_val_vis[:,i], y_pers[:,i])
  balanced_acc_ml[i] = balanced_accuracy_score(y_val_vis[:,i], y_ml[:,i])

In [ ]:
np.set_printoptions(precision=2)
for i in range(future_target):
  print('Precision at prediction step {}: {}'.format(i+1, precision_ml[:,i]))
  print('Recall at prediction step {}: {}'.format(i+1, recall_ml[:,i]))
  print('F1-score at prediction step {}: {}'.format(i+1, f1_ml[:,i]))
  print('Balanced accuracy at prediction step {}: {}'.format(i+1, np.round(100*balanced_acc_ml[i])/100.0))

In [ ]:
for i in range(future_target):
  print('Precision at prediction step {}: {}'.format(i+1, precision_pers[:,i]))
  print('Recall at prediction step {}: {}'.format(i+1, recall_pers[:,i]))
  print('F1-score at prediction step {}: {}'.format(i+1, f1_pers[:,i]))
  print('Balanced accuracy at prediction step {}: {}'.format(i+1, np.round(100*balanced_acc_pers[i])/100.0))

In [ ]:
for i in range(future_target):
  print('Classification summary ML model at forecast hour {}:\n'.format(i+1))
  print(classification_report(y_val_vis[:,i], y_ml[:,i], labels=[0,1,2,3,4,5,6,7]))
  print('Confusion matrix:\n')
  print(confusion_matrix(y_val_vis[:,i], y_ml[:,i], labels=[0,1,2,3,4,5,6,7]))
  print('\n-----------------------------------------------------------')
  print('-----------------------------------------------------------\n')
  print('Classification summary Persistence at forecast hour {}:\n'.format(i+1))
  print(classification_report(y_val_vis[:,i], y_pers[:,i], labels=[0,1,2,3,4,5,6,7]))
  print('Confusion matrix:\n')
  print(confusion_matrix(y_val_vis[:,i], y_pers[:,i], labels=[0,1,2,3,4,5,6,7]))  
  print('\n')

In [ ]:
# Classification summary ML model at forecast hour 1:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.58      0.58      0.58        72
#            2       0.47      0.49      0.48       114
#            3       0.00      0.00      0.00        69
#            4       0.28      0.27      0.28       116
#            5       0.45      0.43      0.44       314
#            6       0.59      0.59      0.59       745
#            7       0.88      0.92      0.90      1905

#     accuracy                           0.74      3337
#    macro avg       0.41      0.41      0.41      3337
# weighted avg       0.72      0.74      0.73      3337

# Confusion matrix:

# [[   0    2    0    0    0    0    0    0]
#  [   0   42   16    0    4    4    6    0]
#  [   0   20   56    0   14   14    9    1]
#  [   0    2   21    0   27   10    9    0]
#  [   0    4   13    0   31   48   20    0]
#  [   0    2   10    0   24  135  121   22]
#  [   0    0    2    0    8   74  442  219]
#  [   0    0    1    0    1   12  138 1753]]

# -----------------------------------------------------------
# -----------------------------------------------------------

# Classification summary Persistence at forecast hour 1:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.54      0.54      0.54        72
#            2       0.46      0.46      0.46       114
#            3       0.28      0.28      0.28        69
#            4       0.30      0.30      0.30       116
#            5       0.44      0.44      0.44       314
#            6       0.59      0.59      0.59       745
#            7       0.89      0.89      0.89      1905

#     accuracy                           0.72      3337
#    macro avg       0.44      0.44      0.44      3337
# weighted avg       0.72      0.72      0.72      3337

# Confusion matrix:

# [[   0    2    0    0    0    0    0    0]
#  [   2   39   14    4    3    6    4    0]
#  [   0   20   53   11   14    8    7    1]
#  [   0    2   19   19   13    8    8    0]
#  [   0    4   15   13   35   37   12    0]
#  [   0    4    8   16   34  137   94   21]
#  [   0    1    3    5   14   90  439  193]
#  [   0    0    2    1    3   28  181 1690]]


# Classification summary ML model at forecast hour 2:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.47      0.39      0.43        72
#            2       0.38      0.33      0.36       114
#            3       0.00      0.00      0.00        69
#            4       0.14      0.06      0.08       116
#            5       0.30      0.22      0.25       314
#            6       0.44      0.44      0.44       745
#            7       0.80      0.90      0.85      1905

#     accuracy                           0.66      3337
#    macro avg       0.32      0.29      0.30      3337
# weighted avg       0.61      0.66      0.63      3337

# Confusion matrix:

# [[   0    1    0    0    0    1    0    0]
#  [   0   28   16    0    3   10   11    4]
#  [   0   17   38    0    6   21   21   11]
#  [   0    4   12    0   12   16   20    5]
#  [   0    5   13    0    7   41   39   11]
#  [   0    4   11    0   14   68  164   53]
#  [   0    0    7    0    6   52  331  349]
#  [   0    0    3    0    1   20  163 1718]]

# -----------------------------------------------------------
# -----------------------------------------------------------

# Classification summary Persistence at forecast hour 2:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.39      0.39      0.39        72
#            2       0.31      0.31      0.31       114
#            3       0.10      0.10      0.10        69
#            4       0.14      0.14      0.14       116
#            5       0.32      0.32      0.32       314
#            6       0.42      0.42      0.42       745
#            7       0.81      0.81      0.81      1905

#     accuracy                           0.62      3337
#    macro avg       0.31      0.31      0.31      3337
# weighted avg       0.62      0.62      0.62      3337

# Confusion matrix:

# [[   0    1    0    0    0    1    0    0]
#  [   2   28   16    5    4    6    8    3]
#  [   0   20   35   12   13    9   17    8]
#  [   0    4   15    7   17   11   11    4]
#  [   0    5   16   11   16   34   27    7]
#  [   0    8   14   17   27   99  106   43]
#  [   0    3   11   11   23   93  316  288]
#  [   0    3    7    6   16   61  260 1552]]


# Classification summary ML model at forecast hour 3:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.37      0.26      0.31        72
#            2       0.30      0.18      0.22       114
#            3       0.00      0.00      0.00        69
#            4       0.14      0.03      0.06       116
#            5       0.18      0.06      0.09       314
#            6       0.34      0.33      0.33       745
#            7       0.73      0.91      0.81      1905

#     accuracy                           0.61      3337
#    macro avg       0.26      0.22      0.23      3337
# weighted avg       0.54      0.61      0.56      3337

# Confusion matrix:

# [[   0    1    0    0    0    0    1    0]
#  [   0   19   12    0    2    6   26    7]
#  [   0   15   20    0    6   11   43   19]
#  [   0    2    6    0    5   14   29   13]
#  [   0    7    9    0    4   14   59   23]
#  [   0    5   10    0    5   18  174  102]
#  [   0    3    7    0    6   19  245  465]
#  [   0    0    2    0    0   18  145 1740]]

# -----------------------------------------------------------
# -----------------------------------------------------------

# Classification summary Persistence at forecast hour 3:

#               precision    recall  f1-score   support

#            0       0.00      0.00      0.00         2
#            1       0.31      0.31      0.31        72
#            2       0.22      0.22      0.22       114
#            3       0.10      0.10      0.10        69
#            4       0.13      0.13      0.13       116
#            5       0.27      0.27      0.27       314
#            6       0.37      0.37      0.37       745
#            7       0.77      0.77      0.77      1905

#     accuracy                           0.57      3337
#    macro avg       0.27      0.27      0.27      3337
# weighted avg       0.57      0.57      0.57      3337

# Confusion matrix:

# [[   0    1    0    0    0    0    1    0]
#  [   1   22   15    4    6    8   11    5]
#  [   0   17   25   14    9   11   24   14]
#  [   0    3   10    7   14   14   11   10]
#  [   1    5   15    9   15   24   31   16]
#  [   0   11   18   10   22   84  101   68]
#  [   0    5   15   15   27   80  272  331]
#  [   0    8   16   10   23   93  294 1461]]


# /usr/local/lib/python3.6/dist-packages/sklearn/metrics/_classification.py:1272: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
#   _warn_prf(average, modifier, msg_start, len(result))